# Financial statement modeling

This notebook introduces **financial statement modeling** in finstack-quant using the statements DSL, `ModelBuilder`, and `Evaluator`.

**Prerequisites:** complete `01_foundations/core_types_and_money.ipynb`, `01_foundations/dates_calendars_schedules.ipynb`, `01_foundations/market_data_and_curves.ipynb`, and `01_foundations/math_toolkit.ipynb` so you are comfortable with periods, numeric data, and running cells top-to-bottom in this series.

## Concept: models as specs + formulas

A **financial model** here is a declarative **specification**: named line items (*nodes*) over a **period lattice** (for example quarterly `2025Q1` … `2025Q4`). Some nodes hold **explicit values** (actuals or assumptions). Others are **computed** from a small **DSL** (e.g. `revenue - cogs`). You can mark where **actuals** end and **forecast** periods begin so downstream tools can treat periods consistently.

Workflow:

1. Configure periods and nodes with `ModelBuilder`.
2. Call `build()` to get an immutable `FinancialModelSpec` (serializable to JSON).
3. Run `Evaluator().evaluate(spec)` to get a `StatementResult` (per-node, per-period numbers), then export to pandas or JSON.

Optional helpers include **formula parse/validate**, **forecast method** tokens (for richer specs), and **normalization** utilities that return JSON or row dicts for adjustment workflows.

### `ModelBuilder`: fluent API

Call `periods(range, actuals_until)` once, then add nodes with `value(...)` (explicit scalars per period) or `compute(...)` (DSL expression). Finish with `build()`.

In [1]:
import pandas as pd

from finstack_quant.statements import (
    ModelBuilder,
    Evaluator,
    FinancialModelSpec,
    StatementResult,
    parse_formula,
    validate_formula,
    ForecastMethod,
    ForecastSpec,
    NodeType,
    NodeId,
    NumericMode,
    NormalizationConfig,
    normalize,
    MetricRegistry,
)

b = ModelBuilder("my-pl-model")
b.periods("2025Q1..Q4", "2025Q2")  # Q1–Q2 actuals, Q3–Q4 forecast
b.value(
    "revenue",
    [("2025Q1", 100.0), ("2025Q2", 110.0), ("2025Q3", 115.0), ("2025Q4", 120.0)],
)
b.value(
    "cogs",
    [("2025Q1", 60.0), ("2025Q2", 65.0), ("2025Q3", 68.0), ("2025Q4", 72.0)],
)
b.compute("gross_profit", "revenue - cogs")
b.compute("gross_margin", "gross_profit / revenue")
b.value(
    "opex",
    [("2025Q1", 20.0), ("2025Q2", 22.0), ("2025Q3", 23.0), ("2025Q4", 24.0)],
)
b.compute("ebitda", "gross_profit - opex")
spec = b.build()

print("NodeType.calculated:", NodeType.calculated())
print("NodeId:", NodeId("revenue"))
print("NumericMode.float64:", NumericMode.float64())

print("spec.id:", spec.id)
print("period_count:", spec.period_count)
print("node_count:", spec.node_count)
print("node_ids():", spec.node_ids())
print("has_node(revenue):", spec.has_node("revenue"))
print("schema_version:", spec.schema_version)

NodeType.calculated: NodeType(Calculated)
NodeId: revenue
NumericMode.float64: NumericMode(Float64)
spec.id: my-pl-model
period_count: 4
node_count: 6
node_ids(): ['revenue', 'cogs', 'gross_profit', 'gross_margin', 'opex', 'ebitda']
has_node(revenue): True
schema_version: 2


### `FinancialModelSpec`: metadata and JSON

The spec is the portable definition of the model. Serialize with `to_json()` and reload with `from_json(...)` for storage, APIs, or scenario tooling.

In [2]:
json_str = spec.to_json()
print("to_json length:", len(json_str))
print("to_json prefix:", json_str[:180].replace("\n", " "), "...")
spec2 = FinancialModelSpec.from_json(json_str)
print("from_json id:", spec2.id)
print("round-trip node_count match:", spec2.node_count == spec.node_count)

to_json length: 1095
to_json prefix: {"id":"my-pl-model","periods":[{"id":"2025Q1","start":"2025-01-01","end":"2025-04-01","is_actual":true},{"id":"2025Q2","start":"2025-04-01","end":"2025-07-01","is_actual":true},{"i ...
from_json id: my-pl-model
round-trip node_count match: True


### `Evaluator` and `StatementResult`

`Evaluator().evaluate(spec)` materializes every computed node for every period. `StatementResult` exposes scalars, node slices, counts, timing metadata, JSON round-trip, and **pandas** exports.

In [3]:
evaluator = Evaluator()
result = evaluator.evaluate(spec)
print(result)
print("sample get:", result.get("revenue", "2025Q1"))
print("ebitda 2025Q4:", result.get("ebitda", "2025Q4"))

StatementResult(nodes=6, periods=4)
sample get: 100.0
ebitda 2025Q4: 24.0


### Slicing results and pandas export

Use `get` / `get_node` for ad hoc inspection, or `to_pandas_wide()` / `to_pandas_long()` for analysis. Below, `to_json` / `from_json` checks lossless round-trip for the numeric result set.

In [4]:
print("get(revenue, 2025Q1):", result.get("revenue", "2025Q1"))
print("get_node(revenue):", result.get_node("revenue"))
print("node_ids():", result.node_ids())
print("node_count:", result.node_count, "num_periods:", result.num_periods)
print("eval_time_ms:", result.eval_time_ms, "warning_count:", result.warning_count)
print("get_scalar revenue Q1:", result.get_scalar("revenue", "2025Q1"))
out_json = result.to_json()
print("result.to_json length:", len(out_json))
result2 = StatementResult.from_json(out_json)
print("round-trip get ebitda Q1:", result2.get("ebitda", "2025Q1"))
wide_df = result.to_pandas_wide()
print(wide_df)
long_df = result.to_pandas_long()
print(long_df)

get(revenue, 2025Q1): 100.0
get_node(revenue): {'2025Q1': 100.0, '2025Q2': 110.0, '2025Q3': 115.0, '2025Q4': 120.0}
node_ids(): ['opex', 'cogs', 'revenue', 'gross_profit', 'ebitda', 'gross_margin']
node_count: 6 num_periods: 4
eval_time_ms: 0 warning_count: 0
get_scalar revenue Q1: 100.0
result.to_json length: 762
round-trip get ebitda Q1: 20.0
period_id     2025Q1      2025Q2      2025Q3  2025Q4
opex            20.0   22.000000   23.000000    24.0
cogs            60.0   65.000000   68.000000    72.0
revenue        100.0  110.000000  115.000000   120.0
gross_profit    40.0   45.000000   47.000000    48.0
ebitda          20.0   23.000000   24.000000    24.0
gross_margin     0.4    0.409091    0.408696     0.4
         node_id  period       value value_money currency value_type
0           opex  2025Q1   20.000000        None     None     scalar
1           opex  2025Q2   22.000000        None     None     scalar
2           opex  2025Q3   23.000000        None     None     scalar
3     

### DSL: `parse_formula` and `validate_formula`

Computed nodes use the statements expression language. Parse to inspect the compiled form, or validate strings before attaching them to a builder.

In [5]:
parsed = parse_formula("revenue - cogs")
print("parsed:", parsed)
is_valid = validate_formula("revenue * (1 + growth_rate)")
print("validate_formula:", is_valid)
try:
    validate_formula("revenue +")
except ValueError as err:
    print("invalid formula (expected):", err)

parsed: BinOp { op: Sub, left: NodeRef(NodeId("revenue")), right: NodeRef(NodeId("cogs")) }
validate_formula: True
invalid formula (expected): Formula parse error: unexpected input at line 1 col 9: '+'


### `ForecastMethod`

Forecast methods are typed tokens used in richer modeling flows (growth curves, forward fill, overrides, etc.). Common factories:

In [6]:
print("forward_fill:", ForecastMethod.forward_fill())
print("growth_pct:", ForecastMethod.growth_pct())
print("curve_pct:", ForecastMethod.curve_pct())

forward_fill: ForecastMethod(ForwardFill)
growth_pct: ForecastMethod(GrowthPct)
curve_pct: ForecastMethod(CurvePct)


### `ForecastSpec` and builder `.forecast`

`ForecastSpec` pairs a `ForecastMethod` with parameters. Attach via `ModelBuilder.forecast(node, spec)`. This is how richer projection rules (growth curves, stochastic, external series) are declared.


In [7]:
fs = ForecastSpec.growth(0.05)
print("growth ForecastSpec:", fs)

b2 = ModelBuilder("forecast-demo")
b2.periods("2025Q1..Q3", "2025Q1")
b2.value("revenue", [("2025Q1", 100.0)])
b2.forecast("revenue", ForecastSpec.growth(0.10))
fs_spec = b2.build()
fs_res = Evaluator().evaluate(fs_spec)
print("revenue 2025Q2 (growth 10%):", fs_res.get("revenue", "2025Q2"))


growth ForecastSpec: ForecastSpec(method=GrowthPct, params=1)
revenue 2025Q2 (growth 10%): 110.00000000000001


### `MixedNodeBuilder` (value + forecast + formula in one node)

Use `b.mixed(node).values(...).forecast(...).formula(...).build()` for nodes that combine actuals, a forecast rule, and a formula fallback.


In [8]:
bm = ModelBuilder("mixed-demo")
bm.periods("2025Q1..Q3", "2025Q1")
mx = bm.mixed("ebitda")
mx.values([("2025Q1", 40.0)])
mx.forecast(ForecastSpec.growth(0.05))
mx.formula("revenue - cogs - opex")
bm2 = mx.build()
mixed_spec = bm2.build()
print("mixed node ids:", mixed_spec.node_ids())


mixed node ids: ['ebitda']


### `NormalizationConfig` and `normalize`

`normalize(result, config)` returns a **JSON** string describing per-period base values, adjustments, and final values. Parse it with `json.loads` to get Python dict rows; wrap those in pandas for display.

This cell rebuilds a tiny model so it runs even if you skipped earlier sections.

In [9]:
import json

nb = ModelBuilder("norm-demo")
nb.periods("2025Q1..Q1", None)
nb.value("ebitda", [("2025Q1", 50.0)])
norm_spec = nb.build()
norm_result = Evaluator().evaluate(norm_spec)
config = NormalizationConfig("ebitda")
norm_json = normalize(norm_result, config)
print("normalize JSON:", norm_json)
rows = json.loads(norm_json)
print("rows:", rows)
print(pd.DataFrame(rows))

normalize JSON: [{"period":"2025Q1","base_value":50.0,"adjustments":[],"final_value":50.0}]
rows: [{'period': '2025Q1', 'base_value': 50.0, 'adjustments': [], 'final_value': 50.0}]
   period  base_value adjustments  final_value
0  2025Q1        50.0          []         50.0


### Value scalars vs Money, MetricRegistry, and NumericMode

`value_scalar` / `value_money` are explicit. `MetricRegistry` provides reusable metrics. Pass `NumericMode.decimal()` to `Evaluator` for currency-exact math.


In [10]:
from finstack_quant.core.money import Money
from finstack_quant.statements import FinancialModelSpec

bv = ModelBuilder("value-money-demo")
bv.periods("2025Q1..Q1", None)
bv.value_scalar("units", [("2025Q1", 1000.0)])
bv.value_money("price", [("2025Q1", Money(12.50, "USD"))])
bv.compute("revenue_money", "units * price")
reg = MetricRegistry.with_builtins()
print("registry has ebitda?:", reg.has("ebitda"))

vm_spec = bv.build()
print("NumericMode tokens:", NumericMode.float64(), NumericMode.decimal())
vm_res = Evaluator().evaluate(vm_spec)
print("revenue_money (default eval):", vm_res.get("revenue_money", "2025Q1"))

# explicit JSON round-trip for model + result
spec_j = vm_spec.to_json()
spec2 = FinancialModelSpec.from_json(spec_j)
res2 = Evaluator().evaluate(spec2)
print("round-trip get revenue_money:", res2.get("revenue_money", "2025Q1"))


registry has ebitda?: False
NumericMode tokens: NumericMode(Float64) NumericMode(Decimal)
revenue_money (default eval): 12500.0
round-trip get revenue_money: 12500.0


## Mini-example: quarterly P&L (4 quarters)

Build a compact **P&L**: revenue, COGS, gross profit, OpEx, D&A, EBITDA, EBIT, interest, EBT, taxes (25% of EBT), and net income. Periods `2025Q1`–`2025Q4` with **actuals through `2025Q2`** and forecast in Q3–Q4. Evaluate and print a **pandas wide** table.

In [11]:
p = ModelBuilder("acme-pl-2025")
p.periods("2025Q1..Q4", "2025Q2")
p.value(
    "revenue",
    [("2025Q1", 100.0), ("2025Q2", 108.0), ("2025Q3", 112.0), ("2025Q4", 118.0)],
)
p.value(
    "cogs",
    [("2025Q1", 55.0), ("2025Q2", 58.0), ("2025Q3", 60.0), ("2025Q4", 63.0)],
)
p.compute("gross_profit", "revenue - cogs")
p.value(
    "opex",
    [("2025Q1", 18.0), ("2025Q2", 19.0), ("2025Q3", 20.0), ("2025Q4", 21.0)],
)
p.value(
    "da",
    [("2025Q1", 4.0), ("2025Q2", 4.0), ("2025Q3", 4.5), ("2025Q4", 5.0)],
)
p.compute("ebitda", "gross_profit - opex")
p.compute("ebit", "ebitda - da")
p.value(
    "interest",
    [("2025Q1", 2.0), ("2025Q2", 2.0), ("2025Q3", 2.2), ("2025Q4", 2.3)],
)
p.compute("ebt", "ebit - interest")
p.compute("taxes", "max(ebt, 0) * 0.25")
p.compute("net_income", "ebt - taxes")
pnl_spec = p.build()
pnl_result = Evaluator().evaluate(pnl_spec)
pnl_wide = pnl_result.to_pandas_wide()
print("P&L wide:")
print(pnl_wide)

P&L wide:
period_id     2025Q1  2025Q2   2025Q3   2025Q4
interest        2.00    2.00    2.200    2.300
da              4.00    4.00    4.500    5.000
opex           18.00   19.00   20.000   21.000
cogs           55.00   58.00   60.000   63.000
revenue       100.00  108.00  112.000  118.000
gross_profit   45.00   50.00   52.000   55.000
ebitda         27.00   31.00   32.000   34.000
ebit           23.00   27.00   27.500   29.000
ebt            21.00   25.00   25.300   26.700
taxes           5.25    6.25    6.325    6.675
net_income     15.75   18.75   18.975   20.025


## Takeaways

- **`ModelBuilder`** is the ergonomic way to define periods, inputs, and `compute` nodes using the statements **DSL**.
- **`FinancialModelSpec`** is the serializable blueprint (`to_json` / `from_json`), separate from evaluated numbers.
- **`Evaluator`** turns a spec into a **`StatementResult`**, addressable by node and period, exportable to **JSON** or **pandas** (`to_pandas_wide`, `to_pandas_long`).
- **`parse_formula` / `validate_formula`** help you debug and gate formulas before they enter a model.
- **`ForecastMethod`** tokens describe projection styles for more advanced specs.
- **`normalize`** supports EBITDA-style adjustment workflows from an evaluated result; it returns JSON that you can parse with `json.loads` into dict rows.

Run all cells in order when exploring; the walkthrough reuses `spec` and `result`, while the normalization and P&L sections are self-contained.

### Named mixed-node builder

`ModelBuilder.mixed(...)` returns a `MixedNodeBuilder`, which collects actual values, forecast rules, and formula fallback before `build()` returns the parent model builder.

In [12]:
from finstack_quant.statements import MixedNodeBuilder

print("MixedNodeBuilder type:", MixedNodeBuilder.__name__)
print("mx is MixedNodeBuilder:", isinstance(mx, MixedNodeBuilder))

MixedNodeBuilder type: MixedNodeBuilder
mx is MixedNodeBuilder: True
